# 1.4 算子开发的基本概念
前文已围绕算子与 CANN 架构的关联完成阐述，本节正式聚焦 Ascend C 算子开发核心，从**定义、开发场景、抽象硬件架构**三个维度，拆解开发必备的基础概念，为后续实操搭建完整的知识框架。

本节学习大纲：
- 什么是 Ascend C
- 算子开发场景
- 抽象硬件架构
- 核函数与 Tiling
- 算子开发全流程


## 一、什么是 Ascend C
Ascend C 是 CANN 生态下面向昇腾 AI 处理器的专用算子开发语言，原生兼容 C 与 C++ 标准规范。基于 Ascend C 编写的算子程序，经毕昇编译器编译和运行时调度后，可直接在昇腾 NPU 上高效执行。

Ascend C 秉持**开放芯片完备编程能力，支撑极致性能**的设计理念，构建了多层级 API 体系，让开发者能在开发效率与运行性能之间灵活权衡：

- **语言扩展层 C API**：纯 C 接口，提供数组内存分配、基于指针的计算接口，延续业界熟悉的 C 编程体验，并完整开放芯片能力。Ascend 950PR/Ascend 950DT 新增 SIMT、SIMD/SIMT 混合编程能力。
- **基础 API**：以单指令为抽象的 C++ 类库，一般基于 Tensor 编程，并通过 Layout 完善 Tensor 编程体验。Layout（布局）用于描述数据在片上存储中的排布方式（如 ND、NZ、ZN 等），使 Tensor 在数据搬运时自动完成格式转换，开发者无需手动处理排布细节。
  - **为什么 Layout 能实现自动格式转换？** 关键在于搬运 API 同时知道源端和目标端的 Layout。例如矩阵乘的输入在 Global Memory 中按 ND（自然行优先）排布，但 Cube 计算单元要求 L0A/L0B 按 NZ（分块列优先）排布。开发者在创建 Tensor 时分别指定 `NDLayout` 和 `NZLayout`，调用 `Copy` 时 API 内部根据两种 Layout 自动生成地址重排指令，一步完成搬运+排布转换。若不用 Layout，开发者需手动计算每个元素的搬运地址和目标偏移，代码量大且容易出错。
- **高阶 API**：对单核常见算法进行抽象与封装，提供开箱即用的公共算法实现。
- **算子模板库**：基于模板提供算子的完整实现参考，降低 Tiling 开发复杂度，支持用户自定义扩展。
- **Python 前端 PyAsc**：基于 Python 原生接口，提供芯片底层完备编程能力，将逐步引入 Layout 化 Tensor 编程、SIMT 编程等，实现用 Python 开发高性能算子。

<img src="./images/architecture_ascenc.png">


## 二、算子开发场景
在实际工程中，算子开发主要面向以下场景（包括但不限于）：

- **自定义算子开发**：框架内置算子库未覆盖的新算子，如新型激活函数、自定义 Attention 变体。
- **性能优化**：框架已有算子性能不达预期，需针对特定 shape/数据类型手写高性能实现。
- **融合算子**：将多个小算子融合为一个大算子，减少中间结果回写 Global Memory 与 Kernel 启动开销。如 FlashAttention 融合了 MatMul+Scale+Mask+Softmax+Dropout。
- **动态 shape 适配**：框架算子仅支持静态 shape，需开发支持任意 shape 的动态算子。



## 三、抽象硬件架构
Ascend C 对昇腾硬件做了抽象封装，开发者面对的是统一的**抽象硬件架构**，而非具体芯片细节。该抽象架构屏蔽了不同硬件之间的差异，显著降低开发门槛。

<img src="./images/ai_core_abstract_architecture.png">

AI Core 中包含三类核心组件：

| 组件类型 | 包含 | 说明 |
|---|---|---|
| 计算单元 | Cube（矩阵乘加）、Vector（向量计算）、Scalar（标量控制） | Scalar 负责指令调度与地址计算，Cube/Vector 负责高密度并行计算 |
| 存储单元 | Local Memory（片上存储，对应 `LocalTensor`）、Global Memory（片外存储，对应 `GlobalTensor`） | 计算单元只能直接访问 Local Memory，GM 数据需先经 DMA 搬入 |
| 搬运单元 | DMA（Direct Memory Access） | 负责 GM↔Local Memory 之间以及不同层级 Local Memory 之间的数据搬运 |

理解抽象硬件架构，需重点关注以下三个过程：

1. **异步指令流**：Scalar 读取指令序列，将向量计算、矩阵计算、数据搬运指令发射给对应单元的指令队列，Vector/Cube/DMA **异步并行执行**接收到的指令。
2. **同步信号流**：不同指令间可能存在依赖关系，Scalar 下发同步信号确保各单元按正确逻辑顺序执行（如搬运完成后才能计算）。
3. **计算数据流**：DMA 将数据从 GM 搬入 Local Memory → Cube/Vector 完成计算并将结果写回 Local Memory → DMA 将结果搬回 GM，即经典的**搬运-计算-搬运**范式。

这一抽象在 910B/910C/950 上保持一致，确保 Ascend C 算子的跨芯片可移植性。但各芯片的 Local Memory 容量、计算单元能力不同，Tiling 策略需按芯片调整。


## 四、核函数与 Tiling
### 4.1 核函数（Kernel）
核函数是算子的设备侧入口，由 `__global__` 修饰，通过 `<<<>>>` 语法调用，在多核上并行执行。每个核（Block）执行一份 Kernel 代码副本，通过 `asc_get_block_idx()` 区分不同核。

```cpp
__global__ __aicore__ void kernel_name(__gm__ uint8_t *x, ...)
{
    // 算子计算逻辑
}

// Host 侧调用
kernel_name<<<blockDim, l1>>>(x, ...);
```


### 4.2 Tiling 切分
Tiling 是 Ascend C 算子开发的核心概念。由于 Local Memory 容量有限（通常几十 KB），而输入数据可能高达数百 MB，必须将大 shape 数据切分为多个小块（Tile），每次搬入一个 Tile 到 Local Memory 计算，完成后再搬回 Global Memory。

Tiling 的关键考量：
- **片上容量**：Tile 大小不能超过 Local Memory 容量。
- **数据对齐**：Tile 边界需对齐到硬件要求（如 32B），否则影响 DMA 效率。
- **并行度**：Tile 数量应能被核数整除，充分利用多核并行。
- **计算/搬运重叠**：通过 Double Buffer 技术实现下一 Tile 搬入与当前 Tile 计算的流水重叠。


<img src="./images/single_core_data_slicing_schematic.png">


> 上图展示了单核数据切分（Tiling）示意：由于 Local Memory 容量有限，需将大 shape 数据切分为多个 Tile，每次搬入一个 Tile 到片上存储计算，完成后再搬回 Global Memory。


<img src="./images/data_moving_and_vector_computing_process.png">


> 上图展示了数据搬运与向量计算的典型流程：GM -> UB（搬入）-> Vector 计算 -> UB -> GM（搬回），即搬运-计算-搬运的经典范式。通过 Double Buffer 可实现下一 Tile 搬入与当前 Tile 计算的流水重叠。


## 五、算子开发全流程
Ascend C 算子开发遵循以下标准流程：

1. **算子逻辑设计**：明确输入/输出 Tensor 的 shape/dtype/format，设计计算公式与 Tiling 切分策略。
2. **编写核函数**：实现设备侧计算逻辑，包括数据搬运（DataCopy）、计算指令（Add/Mmad 等）、结果回写。
3. **编写 Host 侧调用**：申请 Global Memory、通过 `<<<>>>` 启动核函数、同步等待完成。
4. **编译**：使用毕昇编译器（`bisheng`）或 cmake 构建系统编译为 NPU 可执行文件。
5. **运行与验证**：生成测试数据，运行算子，与框架（PyTorch/MindSpore）结果对比验证精度。

在 950 上，编译时需指定 `--npu-arch=dav-3510`（或 cmake 中 `CMAKE_ASC_ARCHITECTURES=dav-3510`），编译器会自动适配 950 的双发射 SIMD、NDDMA 等新特性。


---
## 课后练习
请根据本节课程学习内容完成以下题目进行自测，在每题下方的代码框中输入选项字母后运行。判断题 A=正确，B=错误。


**第1题**（判断题）AI Core 的计算单元仅能直接访问 Local Memory，Global Memory 中的数据需先经 DMA 搬入才能参与计算。
- A. 正确
- B. 错误


In [ ]:
q1 = ''  # 填入你的选项，如 'A'，修改后务必运行本单元格（Shift+Enter）
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')


**第2题**（判断题）在 AI Core 内部，Scalar 将指令发射给 Cube/Vector/DMA 后，各单元异步并行执行接收到的指令。
- A. 正确
- B. 错误


In [ ]:
q2 = ''  # 填入你的选项，如 'A'，修改后务必运行本单元格（Shift+Enter）
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')


**第3题**（单选题）在抽象硬件架构中，负责指令调度、地址计算和同步信号下发的单元是？
- A. Cube
- B. Vector
- C. Scalar
- D. DMA


In [ ]:
q3 = ''  # 填入你的选项，如 'A'，修改后务必运行本单元格（Shift+Enter）
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')


**第4题**（单选题）Tiling 切分的根本原因是？
- A. 提高精度
- B. Local Memory 容量有限
- C. 简化代码
- D. 减少核数


In [ ]:
q4 = ''  # 填入你的选项，如 'A'，修改后务必运行本单元格（Shift+Enter）
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')


**第5题**（单选题）关于 SIMD 和 SIMT 的适用场景，以下说法正确的是？
- A. SIMD 适合离散访存，SIMT 适合矩阵乘
- B. SIMD 适合规整高密度计算，SIMT 适合离散访存和复杂分支
- C. 两者适用场景完全相同
- D. SIMT 适合所有场景，SIMD 已被淘汰


In [ ]:
q5 = ''  # 填入你的选项，如 'A'，修改后务必运行本单元格（Shift+Enter）
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')


**第6题**（判断题）Layout 用于描述数据在片上存储中的排布方式，使 Tensor 在搬运时自动完成格式转换，开发者无需手动处理排布细节。
- A. 正确
- B. 错误


In [ ]:
q6 = ''  # 填入你的选项，如 'A'，修改后务必运行本单元格（Shift+Enter）
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')


**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
sys.path.insert(0, './answer')
from grade import grade
grade(globals(), '01.04_answer.txt')
